# Critic diagnostic — отдельный ноутбук

Загружает обученный critic из `/workspace/out/critic_resnet.pt` и прогоняет 4 теста его качества:

1. **Local sensitivity** — главный тест. Q(s, a_demo) vs Q(s, a_demo + ε·noise) при разных ε. Если при ε=0.05 разница ≈ 0 — local refine PA-RL не работает (∇_a Q плоский в окрестности demo).
2. **Per-dimension probe** — какие из 7 dims критик «видит». Для x/y/z ожидаем симметричный inverted-U (demo в локальном максимуме). Для gripper — вероятна асимметрия, что объясняет насыщение в local refine без freeze.
3. **Gradient norm** — насколько большой `‖∇_a Q‖` в окрестности demo. Прямая мера usefulness local refine с `η=3e-4`.
4. **Q-spread** среди шумных кандидатов — будет ли в PA-RL осмысленный categorical-sampling.

Запуск занимает 10-30 секунд (загрузка + один forward на 64-batch). Не требует SmolVLA, env-а или rollout-cache — только critic + image cache + state cache.


## 1. Setup — те же константы, что в critic_resnet.ipynb

In [ ]:
import os, math, time, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# Константы (СОВПАДАЮТ с critic_resnet.ipynb после патчей)
RAW_IMG_SIZE     = 256
CRITIC_IMG_SIZE  = 128
ACTION_DIM       = 7
STATE_DIM        = 8

CRITIC_PATH      = "/workspace/out/critic_resnet.pt"
IMG_CACHE_PATH   = "/workspace/data/img_cache_critic_256.npz"
ROLLOUT_CACHE    = "/workspace/data/rollout_cache.npz"
TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"
HF_DATASET       = "k1000dai/libero-spatial"

print(f"device: {device}")
print(f"CRITIC_IMG_SIZE: {CRITIC_IMG_SIZE}")
print(f"CRITIC_PATH: {CRITIC_PATH}")

## 2. Загрузка demo cache + rollouts (для diagnostic batch)

In [ ]:
# Demo cache из k1000dai/libero-spatial (по тому же протоколу, что в critic_resnet.ipynb)
print("Загружаем demo cache…")
if os.path.exists(IMG_CACHE_PATH):
    data = np.load(IMG_CACHE_PATH, allow_pickle=True)
    img1_cache    = data["img1"]
    img2_cache    = data["img2"]
    state_cache   = data["state"]
    action_cache  = data["action"]
    episode_cache = data["episode"]
    frame_cache   = data["frame"]
    print(f"  loaded from cache: {len(action_cache)} frames")
else:
    raise RuntimeError(f"img_cache не найден по пути {IMG_CACHE_PATH}. "
                       f"Сначала запусти critic_resnet.ipynb cell 4.")

# State normalization (используем те же mean/std, что в обучении)
state_mean = state_cache.mean(axis=0)
state_std  = state_cache.std(axis=0) + 1e-6
print(f"state_mean: {state_mean.round(3)}")

# Подключаем rollouts если есть (необязательно — diag всё равно работает)
if os.path.exists(ROLLOUT_CACHE):
    print(f"\nЗагружаем rollouts из {ROLLOUT_CACHE}")
    rdata = np.load(ROLLOUT_CACHE, allow_pickle=True)
    rollout_img1     = rdata["img1"]
    rollout_img2     = rdata["img2"]
    rollout_state    = rdata["state"]
    rollout_action   = rdata["action"]
    rollout_episode  = rdata["episode"]
    rollout_ep_succ  = dict(rdata["ep_success"].item())
    n_succ_r = sum(rollout_ep_succ.values())
    print(f"  rollouts: {len(rollout_action)} frames, "
          f"{len(rollout_ep_succ)} episodes ({n_succ_r} success / "
          f"{len(rollout_ep_succ)-n_succ_r} fail)")
    
    # Склейка для диагностики
    diag_img1   = np.concatenate([img1_cache,   rollout_img1], axis=0)
    diag_img2   = np.concatenate([img2_cache,   rollout_img2], axis=0)
    diag_state  = np.concatenate([state_cache,  rollout_state], axis=0)
    diag_action = np.concatenate([action_cache, rollout_action], axis=0)
else:
    print("\n(rollout cache не найден — диагностика только на demo)")
    diag_img1   = img1_cache
    diag_img2   = img2_cache
    diag_state  = state_cache
    diag_action = action_cache

diag_state_norm = ((diag_state - state_mean) / state_std).astype(np.float32)
print(f"\nFor diagnostic: {len(diag_action)} samples available")

## 3. Архитектура critic-а (та же, что в critic_resnet.ipynb)

In [ ]:
from torchvision.models import resnet18

# ── ImageEncoder ──
class ImageEncoder(nn.Module):
    """ResNet-18 from scratch → 512-d feature."""
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    def forward(self, img):
        return self.backbone(img)


# ── CriticHead с action embedding (точно как в critic_resnet.ipynb) ──
class CriticHead(nn.Module):
    def __init__(self, obs_dim, action_dim=ACTION_DIM, hidden=512, action_emb_dim=128):
        super().__init__()
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
        )
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    """Image encoders + state MLP + 2 critic heads + obs_norm."""
    def __init__(self, state_dim=STATE_DIM, action_dim=ACTION_DIM, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()
        self.enc2 = ImageEncoder()
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        self.obs_dim = 512 + 512 + state_hidden
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)
        e2 = self.enc2(img2)
        s  = self.state_mlp(state)
        obs = torch.cat([e1, e2, s], dim=-1)
        return self.obs_norm(obs)
    
    def forward(self, img1, img2, state, action):
        obs = self.encode(img1, img2, state)
        return self.q1(obs, action), self.q2(obs, action)


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


# Загружаем checkpoint
critic = CriticEnsemble().to(device)
if not os.path.exists(CRITIC_PATH):
    raise RuntimeError(f"Critic checkpoint не найден: {CRITIC_PATH}")

ckpt = torch.load(CRITIC_PATH, map_location=device, weights_only=False)
if isinstance(ckpt, dict) and 'critic' in ckpt:
    critic.load_state_dict(ckpt['critic'])
    print(f"loaded critic from dict-checkpoint: {CRITIC_PATH}")
else:
    critic.load_state_dict(ckpt)
    print(f"loaded critic from raw state_dict: {CRITIC_PATH}")
critic.eval()
print(f"  total params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M")
print(f"  obs_dim: {critic.obs_dim}")

## 4. Diagnostic: 4 теста + сводный вердикт

In [ ]:
@torch.no_grad()
def diagnose_critic(critic, batch_obs_critic, batch_action_demo):
    """Полная диагностика. Возвращает dict с метриками."""
    B = batch_obs_critic.shape[0]
    device = batch_obs_critic.device
    dim_names = ['x', 'y', 'z', 'rx', 'ry', 'rz', 'gripper']
    
    print("="*72)
    print(f"  Critic diagnostic on {B} samples")
    print("="*72)
    
    q_demo = critic.q1(batch_obs_critic, batch_action_demo)
    print(f"\nQ(s, a_demo): mean={q_demo.mean().item():+.2f}, std={q_demo.std().item():.2f}")
    
    # ── Test 1: Local sensitivity ──
    print("\n[1] Local sensitivity (Q(demo) vs Q(demo + noise)):")
    print(f"  {'eps':>10} | {'mean dQ':>10} | {'std dQ':>10} | {'frac better':>12}")
    sensitivity = {}
    for eps in [0.01, 0.025, 0.05, 0.1, 0.2, 0.5, 1.0]:
        noise = torch.randn_like(batch_action_demo) * eps
        a_pert = (batch_action_demo + noise).clamp(-1, 1)
        q_pert = critic.q1(batch_obs_critic, a_pert)
        dq = q_demo - q_pert
        frac_better = (dq > 0).float().mean().item()
        sensitivity[eps] = (dq.mean().item(), dq.std().item(), frac_better)
        print(f"  {eps:>10.3f} | {dq.mean().item():>+10.3f} | "
              f"{dq.std().item():>10.3f} | {frac_better:>12.1%}")
    
    # ── Test 2: Per-dimension probe ──
    print("\n[2] Per-dimension probe (eps=0.1, single-dim shift):")
    print(f"  {'dim':>7} | {'+eps dQ':>10} | {'-eps dQ':>10} | {'symmetric?':>12}")
    per_dim = {}
    for d, name in enumerate(dim_names):
        dq_pos, dq_neg = None, None
        for sign in [+1, -1]:
            shift = torch.zeros_like(batch_action_demo)
            shift[:, d] = sign * 0.1
            q_shifted = critic.q1(batch_obs_critic, (batch_action_demo + shift).clamp(-1, 1))
            d_q = (q_demo - q_shifted).mean().item()
            if sign > 0: dq_pos = d_q
            else:        dq_neg = d_q
        sym = abs(dq_pos - dq_neg) < 0.5
        per_dim[name] = (dq_pos, dq_neg, sym)
        sym_str = "  sym" if sym else "  asym"
        print(f"  {name:>7} | {dq_pos:>+10.3f} | {dq_neg:>+10.3f} | {sym_str:>12}")
    
    # ── Test 3: Gradient norm ──
    print("\n[3] Gradient norm |grad_a Q| (batch distribution):")
    a_req = batch_action_demo.clone().detach().requires_grad_(True)
    with torch.enable_grad():
        q_for_grad = critic.q1(batch_obs_critic, a_req)
        g = torch.autograd.grad(q_for_grad.sum(), a_req)[0]
    g_norms = g.norm(dim=-1)
    p10 = g_norms.kthvalue(max(1, int(B*0.1))).values.item()
    p90 = g_norms.kthvalue(max(1, int(B*0.9))).values.item()
    print(f"  median: {g_norms.median().item():.3f}")
    print(f"  mean:   {g_norms.mean().item():.3f}")
    print(f"  p10:    {p10:.3f}")
    print(f"  p90:    {p90:.3f}")
    g_per_dim = g.abs().mean(dim=0)
    print(f"  per-dim |g|:")
    for d, name in enumerate(dim_names):
        print(f"    {name:>7}: {g_per_dim[d].item():.4f}")
    
    # ── Test 4: Q-spread among 16 perturbed candidates ──
    print("\n[4] Q-discrimination среди N=16 perturbed candidates (eps=0.1):")
    n_cand = 16
    eps = 0.1
    noise = torch.randn(B, n_cand, 7, device=device) * eps
    a_cands = (batch_action_demo.unsqueeze(1) + noise).clamp(-1, 1)
    obs_exp = batch_obs_critic.unsqueeze(1).expand(-1, n_cand, -1)
    q_cands = critic.q1(obs_exp.reshape(B*n_cand, -1),
                        a_cands.reshape(B*n_cand, 7)).reshape(B, n_cand)
    q_std_per_state = q_cands.std(dim=1).mean().item()
    q_range = (q_cands.max(dim=1).values - q_cands.min(dim=1).values).mean().item()
    print(f"  Q std среди {n_cand} candidates на состояние: {q_std_per_state:.3f}")
    print(f"  Q range (max-min) на состояние:               {q_range:.3f}")
    
    # ── Verdict ──
    print("\n" + "="*72)
    print("  ВЕРДИКТ:")
    print("="*72)
    
    eps05_dq = sensitivity[0.05][0]
    if eps05_dq > 0.1:
        v_local = "✓ Critic РАЗЛИЧАЕТ мелкие перестановки демо"
    elif eps05_dq > 0.02:
        v_local = "~ Critic слабо различает мелкие перестановки (граница)"
    else:
        v_local = "✗ Critic НЕ различает мелкие перестановки → local refine не сработает"
    print(f"  [1] Local sensitivity (eps=0.05): dQ={eps05_dq:.3f}")
    print(f"      → {v_local}")
    
    grad_med = g_norms.median().item()
    if grad_med >= 2.0:
        v_grad = "✓ Сильный градиент → local refine с eta=3e-4 будет двигать действия"
    elif grad_med >= 0.5:
        v_grad = "~ Средний градиент → local refine даст небольшой эффект"
    else:
        v_grad = "✗ Малый градиент → local refine практически бесполезен"
    print(f"  [3] Gradient norm median: {grad_med:.3f}")
    print(f"      → {v_grad}")
    
    if q_std_per_state >= 0.3:
        v_spread = f"✓ Q std={q_std_per_state:.2f} → categorical-sampling в PA-RL осмысленно"
    elif q_std_per_state >= 0.1:
        v_spread = f"~ Q std={q_std_per_state:.2f} → на границе argmax/categorical"
    else:
        v_spread = f"✗ Q std={q_std_per_state:.2f} → critic не различает кандидатов в окрестности"
    print(f"  [4] Q-spread (eps=0.1, N=16):")
    print(f"      → {v_spread}")
    
    print()
    if eps05_dq > 0.1 and grad_med >= 0.5 and q_std_per_state >= 0.1:
        print("  ✓ ОБЩИЙ ПРОГНОЗ: PA-RL должен работать выше filtered-BC потолка.")
    elif eps05_dq < 0.02 or grad_med < 0.1:
        print("  ✗ ОБЩИЙ ПРОГНОЗ: PA-RL вырождается в filtered-BC, local refine не даст лифта.")
        print("    Рекомендация: больше failure-rollouts (200+) для усиления action-discrimination.")
    else:
        print("  ~ ОБЩИЙ ПРОГНОЗ: PA-RL даст слабый лифт. Возможно работает, возможно на грани шума.")
    print("="*72)
    
    return {
        "q_demo_mean":      q_demo.mean().item(),
        "sensitivity":      sensitivity,
        "per_dim":          per_dim,
        "grad_norm_median": grad_med,
        "q_std_per_state":  q_std_per_state,
    }

## 5. Запуск диагностики

In [ ]:
# Сэмплируем 64 случайных индекса из combined-cache (demo + rollouts если есть)
N_DIAG = 64
rng = np.random.RandomState(SEED)
idx = rng.choice(len(diag_action), size=N_DIAG, replace=False)

# Готовим тензоры
img1_t = torch.from_numpy(diag_img1[idx]).permute(0,3,1,2).float().to(device) / 255.0
img2_t = torch.from_numpy(diag_img2[idx]).permute(0,3,1,2).float().to(device) / 255.0
state_t   = torch.from_numpy(diag_state_norm[idx]).float().to(device)
action_t  = torch.from_numpy(diag_action[idx]).float().to(device)

print(f"Сэмплировано {N_DIAG} samples (idx range: [{idx.min()}, {idx.max()}])")
print(f"  img1: {tuple(img1_t.shape)}, img2: {tuple(img2_t.shape)}")
print(f"  state: {tuple(state_t.shape)}, action: {tuple(action_t.shape)}")
print(f"  action range: [{action_t.min().item():+.3f}, {action_t.max().item():+.3f}]")

# Encode obs (downscale 256 → 128, как при обучении)
with torch.no_grad():
    img1_d = downscale_only(img1_t, CRITIC_IMG_SIZE)
    img2_d = downscale_only(img2_t, CRITIC_IMG_SIZE)
    obs_d  = critic.encode(img1_d, img2_d, state_t)

# Запуск диагностики
diag_results = diagnose_critic(critic, obs_d, action_t)

## 6. (Опционально) Сравнить критика на demo-only vs rollout-only

Если у тебя есть rollouts — полезно сравнить, как критик ведёт себя на demo-сэмплах vs на rollout-сэмплах. Если на demo gradient большой, а на rollouts — нет, это значит критик переобучен на demo distribution и не генерализует на менее экспертные состояния.

In [ ]:
if os.path.exists(ROLLOUT_CACHE):
    n_demo = len(action_cache)
    n_rollout = len(rollout_action)
    
    # Diag только на demo
    print(f"\n{'#'*72}")
    print(f"# DEMO-only diagnostic ({n_demo} available)")
    print(f"{'#'*72}")
    idx_demo = rng.choice(n_demo, size=N_DIAG, replace=False)
    img1_d_t = torch.from_numpy(img1_cache[idx_demo]).permute(0,3,1,2).float().to(device) / 255.0
    img2_d_t = torch.from_numpy(img2_cache[idx_demo]).permute(0,3,1,2).float().to(device) / 255.0
    state_d_t = torch.from_numpy(((state_cache[idx_demo] - state_mean)/state_std).astype(np.float32)).to(device)
    action_d_t = torch.from_numpy(action_cache[idx_demo]).float().to(device)
    with torch.no_grad():
        obs_demo = critic.encode(downscale_only(img1_d_t), downscale_only(img2_d_t), state_d_t)
    res_demo = diagnose_critic(critic, obs_demo, action_d_t)
    
    # Diag только на rollouts
    print(f"\n{'#'*72}")
    print(f"# ROLLOUT-only diagnostic ({n_rollout} available)")
    print(f"{'#'*72}")
    idx_rol = rng.choice(n_rollout, size=N_DIAG, replace=False)
    img1_r_t = torch.from_numpy(rollout_img1[idx_rol]).permute(0,3,1,2).float().to(device) / 255.0
    img2_r_t = torch.from_numpy(rollout_img2[idx_rol]).permute(0,3,1,2).float().to(device) / 255.0
    state_r_t = torch.from_numpy(((rollout_state[idx_rol] - state_mean)/state_std).astype(np.float32)).to(device)
    action_r_t = torch.from_numpy(rollout_action[idx_rol]).float().to(device)
    with torch.no_grad():
        obs_rol = critic.encode(downscale_only(img1_r_t), downscale_only(img2_r_t), state_r_t)
    res_rol = diagnose_critic(critic, obs_rol, action_r_t)
    
    # Compare
    print(f"\n{'='*72}")
    print(f"  COMPARE demo vs rollout")
    print(f"{'='*72}")
    print(f"  metric                     |    demo    |  rollout   |  delta")
    print(f"  ---------------------------+------------+------------+--------")
    print(f"  Q(s, a) mean               | {res_demo['q_demo_mean']:>+9.2f}  | "
          f"{res_rol['q_demo_mean']:>+9.2f}  | {res_rol['q_demo_mean']-res_demo['q_demo_mean']:>+6.2f}")
    print(f"  grad norm (median)         | {res_demo['grad_norm_median']:>+9.3f}  | "
          f"{res_rol['grad_norm_median']:>+9.3f}  | "
          f"{res_rol['grad_norm_median']-res_demo['grad_norm_median']:>+6.3f}")
    print(f"  Q std среди 16 candidates  | {res_demo['q_std_per_state']:>+9.3f}  | "
          f"{res_rol['q_std_per_state']:>+9.3f}  | "
          f"{res_rol['q_std_per_state']-res_demo['q_std_per_state']:>+6.3f}")
    
    if res_rol['grad_norm_median'] < 0.5 * res_demo['grad_norm_median']:
        print()
        print("  ⚠ Внимание: grad norm на rollouts заметно меньше чем на demo.")
        print("    Это значит, критик переобучен на demo distribution и слабо")
        print("    обобщает на менее экспертные состояния. Для PA-RL это плохо:")
        print("    при семплировании из политики (state coverage shifted from demo)")
        print("    local refine не найдёт сигнала.")
else:
    print("rollout_cache не найден — сравнение пропускается")